In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from maite_datasets.image_classification import MNIST

from dataeval.config import set_seed
from dataeval.data import ClassFilter, Limit, Operation, Relabel, Shuffle, View

set_seed(0)  # For reproducibility

In [3]:
mnist = MNIST("./data", image_set="test", download=True)
print("source dataset size:", len(mnist))
print("source vocabulary:", mnist.metadata.get("index2label"))

source dataset size: 10000
source vocabulary: {0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}


In [4]:
view = View(mnist, operations=[ClassFilter([0, 1]), Limit(500)])

print("view size:", len(view))
print("source is unchanged:", len(mnist))
print("first 10 source indices behind the view:", view.resolve_indices()[:10])

view size: 500
source is unchanged: 10000
first 10 source indices behind the view: [2, 3, 5, 10, 13, 14, 25, 28, 29, 31]


In [5]:
# TEST ASSERTION CELL ###
assert len(view) == 500
assert len(mnist) == 10000

In [6]:
limit_then_filter = View(mnist, operations=[Limit(500), ClassFilter([0, 1])])
filter_then_limit = View(mnist, operations=[ClassFilter([0, 1]), Limit(500)])

print("Limit(500) -> ClassFilter([0, 1]):", len(limit_then_filter), "(0s and 1s *within* the first 500)")
print("ClassFilter([0, 1]) -> Limit(500):", len(filter_then_limit), "(the first 500 0s and 1s)")

Limit(500) -> ClassFilter([0, 1]): 109 (0s and 1s *within* the first 500)
ClassFilter([0, 1]) -> Limit(500): 500 (the first 500 0s and 1s)


In [7]:
# TEST ASSERTION CELL ###
assert len(limit_then_filter) < len(filter_then_limit) == 500

In [8]:
windowed = View(mnist, operations=[Limit(2000), Shuffle(seed=0), Limit(10)])
print("10 random items drawn from the first 2,000:", windowed.resolve_indices())

10 random items drawn from the first 2,000: [1946, 1236, 1380, 1949, 1633, 474, 1815, 935, 470, 1626]


In [9]:
# TEST ASSERTION CELL ###
assert len(windowed) == 10
assert all(i < 2000 for i in windowed.resolve_indices())

In [10]:
digit_names = mnist.metadata.get("index2label", {})
EVEN_ODD = ("even", "odd")
parity = {name: (EVEN_ODD[digit % 2]) for digit, name in digit_names.items()}


def source_digits(v: View) -> list[int]:
    """The original MNIST digits sitting behind a view's items."""
    return sorted({int(np.argmax(mnist[i][1])) for i in v.resolve_indices()})


relabel_first = View(mnist, operations=[Relabel(parity, EVEN_ODD), ClassFilter([0])])
filter_first = View(mnist, operations=[ClassFilter([0]), Relabel(parity, EVEN_ODD)])

print("vocabulary after Relabel:", relabel_first.metadata.get("index2label"))
print()
print("Relabel -> ClassFilter([0]):", len(relabel_first), "images, source digits", source_digits(relabel_first))
print("ClassFilter([0]) -> Relabel:", len(filter_first), "images, source digits", source_digits(filter_first))

vocabulary after Relabel: {0: 'even', 1: 'odd'}

Relabel -> ClassFilter([0]): 4926 images, source digits [0, 2, 4, 6, 8]
ClassFilter([0]) -> Relabel: 980 images, source digits [0]


In [11]:
# TEST ASSERTION CELL ###
assert "index2label" in relabel_first.metadata
assert relabel_first.metadata["index2label"] == {0: "even", 1: "odd"}
assert source_digits(relabel_first) == [0, 2, 4, 6, 8]
assert source_digits(filter_first) == [0]
assert len(relabel_first) > len(filter_first)

In [12]:
class KeepEvery(Operation):
    """Cardinality: keep every nth item of the current selection."""

    def __init__(self, n: int) -> None:
        self.n = n

    def apply(self, view: View) -> None:
        view.selection = view.selection[:: self.n]


class Invert(Operation):
    """Content: invert pixel intensities. Registers a transform, reads no data here."""

    def apply(self, view: View) -> None:
        view.map(lambda datum: (255 - np.asarray(datum[0]), datum[1], datum[2]))


custom = View(mnist, operations=[Limit(100), KeepEvery(10), Invert()])

print("custom view size:", len(custom))
print("source indices:", custom.resolve_indices())
print(
    "a background pixel — source:",
    int(np.asarray(mnist[0][0])[0, 0, 0]),
    "view:",
    int(np.asarray(custom[0][0])[0, 0, 0]),
)

custom view size: 10
source indices: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
a background pixel — source: 0 view: 255


In [13]:
# TEST ASSERTION CELL ###
assert custom.resolve_indices() == list(range(0, 100, 10))
assert np.array_equal(np.asarray(custom[0][0]), 255 - np.asarray(mnist[0][0]))

In [14]:
nested = View(View(mnist, [ClassFilter([0, 1])]), [Limit(25)])

print("operations, innermost first:", nested.operation_groups)
print("original dataset:", type(nested.root).__name__)
print("source indices behind the first 5 items:", nested.resolve_indices()[:5])

operations, innermost first: [[ClassFilter(classes=[0, 1], filter_detections=True)], [Limit(size=25)]]
original dataset: MNIST
source indices behind the first 5 items: [0, 1, 2, 3, 4]


In [15]:
# TEST ASSERTION CELL ###
assert len(nested.operation_groups) == 2
assert nested.root is mnist